# Verifica della pipeline (test)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Alessandro1040/bpm-crnn/blob/main/notebooks/04_test_pipeline.ipynb)

Notebook di controllo: risponde alla domanda "il repo funziona davvero?" in un
ambiente pulito (Colab), **senza scaricare GTZAN**.

Cosa verifica:
0. dipendenze;
1. **28 test unitari** (`pytest`) su modello, metriche e pipeline dati;
2. **smoke test end-to-end** su un mini-dataset *sintetico* creato qui
   (30 loop con BPM esatto -> cache Mel -> 2 epoche di training -> valutazione
   -> inferenza) e controllo che vengano prodotti tutti gli artefatti
   (`best.pth`, `status.json`, `TRAINING_COMPLETE`, `RESULTS.md`,
   `predictions_test.csv`);
3. **coerenza dei pesi pubblicati**: `models/bpm_crnn_gtzan.pt` si carica, esegue
   una forward pass e contiene le stesse metriche di
   `results/gtzan/test_metrics.json` (cioe' quelle del README).

Nel punto 2 il modello e' addestrato per 2 epoche su 30 loop: le sue previsioni
**non** sono una misura di accuratezza - serve a verificare che la catena giri.

In [ ]:
#@title 0. Dipendenze e import
%pip install -q torch librosa soundfile pandas matplotlib tqdm pytest requests

import json
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

print("torch", torch.__version__, "| device:",
      "cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
#@title 1. Clona il repo (e i pesi pubblicati)
import json, os, subprocess, sys, urllib.request
from pathlib import Path

import numpy as np
import torch

REPO_URL = "https://github.com/Alessandro1040/bpm-crnn.git"
ROOT = Path("/content/bpm-crnn") if Path("/content").is_dir() else Path.cwd() / "bpm-crnn"
if not (ROOT / "src" / "infer.py").exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, str(ROOT)], check=True)
sys.path.insert(0, str(ROOT))
print("repo:", ROOT)

CKPT = ROOT / "models" / "bpm_crnn_gtzan.pt"
RAW_URL = "https://raw.githubusercontent.com/Alessandro1040/bpm-crnn/main/models/bpm_crnn_gtzan.pt"
RELEASE_URL = "https://github.com/Alessandro1040/bpm-crnn/releases/download/v1.0-gtzan/best.pth"
if not CKPT.exists():
    try:
        print("scarico i pesi da GitHub (models/bpm_crnn_gtzan.pt, 21 MB) ...")
        urllib.request.urlretrieve(RAW_URL, CKPT)
    except Exception as exc:                       # fallback: release asset (64 MB, con optimizer)
        print("fallback sulla release:", exc)
        urllib.request.urlretrieve(RELEASE_URL, CKPT)
print("pesi:", CKPT, f"{CKPT.stat().st_size/1e6:.1f} MB")

In [ ]:
#@title 2. Test unitari (pytest)
r = subprocess.run([sys.executable, "-m", "pytest", "tests", "-q"],
                   cwd=ROOT, capture_output=True, text=True)
print(r.stdout[-1500:])
TEST_OK = "passed" in r.stdout and r.returncode == 0
print("pytest exit:", r.returncode)

In [ ]:
#@title 3. Mini-dataset sintetico (nessun download)
import soundfile as sf

from src.dataset import (HOP_LENGTH, N_MELS, SR, log_mel, n_frames,
                         synthesize_rhythm)

# mini-dataset sintetico: 30 loop da 30 s con BPM esatto, nessun download
CACHE = ROOT / "data_synth" / "cache"
CACHE.mkdir(parents=True, exist_ok=True)
N_TRACKS = 30
TOTAL = n_frames(30.0)

mels = np.lib.format.open_memmap(CACHE / "mels.npy", mode="w+",
                                 dtype=np.float16, shape=(N_TRACKS, N_MELS, TOTAL))
rows = []
for i in range(N_TRACKS):
    bpm = float(np.exp(np.random.default_rng(200 + i).uniform(np.log(70.0),
                                                              np.log(180.0))))
    y = synthesize_rhythm(bpm, sr=SR, duration=30.0, rng=np.random.default_rng(300 + i))
    mel = log_mel(y, n_mels=N_MELS, hop_length=HOP_LENGTH)
    frames = mel.shape[1]
    if frames < TOTAL:
        mel = np.pad(mel, ((0, 0), (0, TOTAL - frames)),
                     constant_values=float(mel.min()))
    mels[i] = mel[:, :TOTAL]
    rows.append({"track_id": f"synth/{i:03d}", "path": "synthetic", "genre": "synth",
                 "bpm": bpm, "sha1": f"h{i:04d}", "duration": 30.0, "sample_rate": SR,
                 "mel_mean": float(mel.mean()), "mel_std": float(mel.std() + 1e-8),
                 "n_frames": int(min(frames, TOTAL)),
                 "split": "train" if i < 24 else ("val" if i < 27 else "test")})
mels.flush()
del mels
pd.DataFrame(rows).to_csv(CACHE / "index.csv", index=False)
(CACHE / "cache_config.json").write_text(json.dumps(
    {"sr": SR, "n_mels": N_MELS, "n_fft": 2048, "hop_length": HOP_LENGTH,
     "fmax": 8000.0, "duration": 30.0, "n_tracks": N_TRACKS}, indent=2))
print(f"mini-cache pronta: {CACHE}\n{N_TRACKS} loop, BPM "
      f"{min(r['bpm'] for r in rows):.1f}-{max(r['bpm'] for r in rows):.1f}")

In [ ]:
#@title 4. Smoke test: 2 epoche su 30 loop sintetici
SMOKE = ROOT / "runs" / "smoke_colab"
if SMOKE.exists():
    subprocess.run(["rm", "-rf", str(SMOKE)], check=True)

cmd = [sys.executable, "-m", "src.train", "--cache-dir", str(CACHE),
       "--out", str(SMOKE), "--epochs", "2", "--batch-size", "8",
       "--synthetic", "8", "--num-workers", "0", "--log-every", "2"]
print(" ".join(cmd), "\n")
subprocess.run(cmd, cwd=ROOT, check=True)

print("\n--- ultime righe del log ---")
print("\n".join((SMOKE / "train.log").read_text().splitlines()[-6:]))

In [ ]:
#@title 5. Valutazione + inferenza sugli artefatti del run
subprocess.run([sys.executable, "-m", "src.evaluate", "--ckpt", str(SMOKE / "best.pth"),
                "--cache-dir", str(CACHE), "--out-dir", str(SMOKE),
                "--no-plots"], cwd=ROOT, check=True)

from src.infer import BPMExtractor

extractor = BPMExtractor(SMOKE / "best.pth", device="auto", cache_dir=CACHE)
loop = Path("/tmp/loop_verifica.wav")
sf.write(loop, synthesize_rhythm(120.0, sr=22050, duration=12.0,
                                 rng=np.random.default_rng(7)), 22050)
det = extractor.predict(loop, return_details=True)
print(f"\ninferenza sul loop di prova (modello NON addestrato): {det['bpm']:.2f} BPM "
      f"su {det['n_chunks']} finestre - qui conta che l'API risponda, non il valore.")

In [ ]:
#@title 6. Coerenza dei pesi pubblicati e checklist
from src.model import build_model_from_config
from src.utils import load_checkpoint

pesi = load_checkpoint(CKPT, map_location="cpu")
modello = build_model_from_config(pesi.get("arch") or {})
modello.load_state_dict(pesi["state_dict"])
modello.eval()
with torch.no_grad():
    _, logits = modello(torch.zeros(1, 1, N_MELS, 431))

ref = json.loads((ROOT / "results/gtzan/test_metrics.json").read_text())
embedded = pesi.get("test_metrics", {})
storia = pd.read_csv(ROOT / "results/gtzan/history.csv")
best_epoca = int(storia.loc[int(storia["val_MAE"].idxmin()), "epoch"])

controlli = {
    "28 test unitari passati": TEST_OK,
    "training end-to-end (2 epoche)": (SMOKE / "best.pth").exists(),
    "marker TRAINING_COMPLETE": (SMOKE / "TRAINING_COMPLETE").exists(),
    "report RESULTS.md del run": (SMOKE / "RESULTS.md").exists(),
    "metriche test scritte": (SMOKE / "test_metrics.json").exists(),
    "predizioni per traccia": (SMOKE / "predictions_test.csv").exists(),
    "pesi pubblicati caricabili": tuple(logits.shape) == (1, 107, 256),
    "metriche nei pesi == README": bool(embedded) and abs(
        embedded["mean/MAE"] - ref["mean/MAE"]) < 1e-9,
    "epoca migliore coerente": abs(int(pesi.get("epoch", -1)) - best_epoca) <= 1,
}
print(f"{'controllo':<36}{'esito':>8}")
print("-" * 46)
for nome, esito in controlli.items():
    print(f"{nome:<36}{'OK' if esito else 'FALLITO':>8}")
print("\ntutti i controlli superati:", all(controlli.values()))

### Cosa significano questi controlli

* **Test unitari** verificano che:
  - il modello produca le shape attese e si addestri (gradienti finiti);
  - le metriche trattino correttamente gli **errori d'ottava** (60 vs 120 BPM);
  - la pipeline dati sia coerente: augmentation con label aggiornata, loop
    sintetici con BPM esatto, cache Mel, split senza leakage, batch rettangolari.
* **Smoke test**: la stessa catena del modello pubblicato
  (`src.train` -> `src.evaluate` -> `BPMExtractor`) gira da zero su dati
  autocontenuti e produce tutti gli artefatti documentati nel README.
* **Coerenza dei pesi**: il file in `models/` e' quello che ha generato i numeri in
  `results/gtzan/test_metrics.json` (e quindi la tabella del README), e la storia
  di training conferma l'epoca migliore.